In [ ]:
# Silver load for CRM customers: Files/landing/customers/*.csv -> managed Delta table silver_customers
from delta.tables import DeltaTable
from pyspark.sql import Window
from pyspark.sql import functions as F

LANDING_DIR = "Files/landing/customers"
ARCHIVE_DIR = "Files/archive/customers"
TARGET_TABLE = "silver_customers"

# Direct Lake consumer: write V-Order optimized Parquet for this session,
# regardless of the workspace's write-heavy resource profile.
spark.conf.set("spark.sql.parquet.vorder.default", "true")

In [ ]:
csv_files = [f for f in notebookutils.fs.ls(LANDING_DIR) if not f.isDir and f.name.lower().endswith(".csv")]
if not csv_files:
    notebookutils.notebook.exit("no new files")

raw = (spark.read
       .option("header", "true")
       .option("inferSchema", "true")
       .csv([f.path for f in csv_files]))

latest = (raw
          .withColumn("updated_at", F.to_timestamp("updated_at"))
          .withColumn("_rn", F.row_number().over(
              Window.partitionBy("customer_id").orderBy(F.col("updated_at").desc())))
          .filter(F.col("_rn") == 1)
          .drop("_rn"))

In [ ]:
if spark.catalog.tableExists(TARGET_TABLE):
    (DeltaTable.forName(spark, TARGET_TABLE).alias("t")
        .merge(latest.alias("s"), "t.customer_id = s.customer_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
else:
    latest.write.format("delta").mode("overwrite").saveAsTable(TARGET_TABLE)
    spark.sql(f"ALTER TABLE {TARGET_TABLE} SET TBLPROPERTIES ('delta.parquet.vorder.enabled' = 'true')")

In [ ]:
for f in csv_files:
    notebookutils.fs.mv(f.path, f"{ARCHIVE_DIR}/{f.name}", True)

notebookutils.notebook.exit(f"processed {len(csv_files)} file(s)")